<p style="font-size: 30px; font-weight:bold">FnGenesis dll tester</p>  
<br>
<br>
본 테스터는 FnGenesis 엔진 dll 파일이 평가 시스템과 유사한 환경(python)에서 정상적으로 실행되는지 확인할 수 있도록 제작되었습니다. 추가로 개선/수정이 필요한 사항이 있다면 자유롭게 제안 부탁드립니다.

In [10]:
import time
import os
import ctypes
from ctypes import WinDLL, c_wchar_p, c_double, POINTER, byref, windll, wintypes
import json
import subprocess
import psutil

# VS 연결

**dll 프로세스와 연결(Ctrl+Alt+P)** 하여 디버그 할때 아래 PID에 해당하는 Python 프로세스에 연결하면 됩니다. (연결 시에는 코드 형식을 **네이티브 코드**로 설정하셔야 dll 소스코드를 단계 별로 디버깅 가능합니다.

In [11]:
print("PID:", os.getpid())

PID: 24668


# Export 함수 목록 확인

FnGenesis 프로젝트 dll 내 함수들은 extern "C"로 export 되어 있지만, 32비트 환경에서도 호출 함수명을 정상적으로 사용하기 위해서는 **FnGenesis.def** 파일 적용이 필요합니다.

*※ **적용방법**(최초 1회만 필요)*  
*: Visual Studio→프로젝트→속성(Release→win32 or x86)→링커→입력→모듈 정의 파일을 "FnGenesis.def"로 설정*

**dumpbin.exe 경로 설정**  
: Visual Studio 업데이트를 진행하는 경우 경로명의 *"MSVC/**xx.xx.xxxxx**/bin"* 부분을 수정해주어야 합니다.

In [12]:
dumpbin_path = r"C:/Program Files/Microsoft Visual Studio/2022/Professional/VC/Tools/MSVC/14.44.35207/bin/Hostx64/x64/dumpbin.exe"

**dll 함수 목록 출력**  
: FnGenesis.def이 제대로 적용되었는지 확인하기 위해 **dumpbin.exe** 파일을 이용해 export된 함수명들의 목록을 확인할 수 있습니다. **"abckillerFC = _abckillerFC@8"** 형태와 같이 등호 기준 앞족에는 언더바(_)가 남아있지 않아야 합니다.

In [23]:
# dll 경로 설정
#dll_path = "D:/source/repos/MBS_Pricer/x64/Release/MBSPricer_release.dll"
#dll_path = "D:/source/repos/MBS_Pricer/x64/Release/MBSPricer_release_0.27.0.dll"
#dll_path = "D:/source/repos/MBS_Pricer/x64/Debug//MBSPricer.dll"
#dll_path = "D:/source/repos/FNZeroCallable/x64/Debug/IREngine.dll"
dll_path = "D:/source/repos/FnGenesis/x64/Debug/FnGenesis.dll"
#dll_path = "D:/OneDrive - postech.ac.kr/FnPricing/ZeroCallable/최종전달용/IREngine_x64_251222.dll"
#dll_path = "D:/source/repos/FnGenesis/x64/Release/FnGenesis_DCF_x64_251002.dll"

## DLL 실행

In [37]:
# dumpbin.exe 실행 결과 출력 (인코딩 안전)
result = subprocess.run(
    [dumpbin_path, "/EXPORTS", dll_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=False,          # 바이트로 받기
    shell=False          # 불필요한 셸 사용 해제
)

def _decode(data: bytes) -> str:
    for enc in ("cp949", "utf-8", "cp1252", "mbcs", "latin-1"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    return data.decode("latin-1", errors="replace")

out = _decode(result.stdout or b"")
err = _decode(result.stderr or b"")

if result.returncode == 0:
    print("DLL function list:")
    print(out)
else:
    print("Error:", err)

# dll 로딩

dllFnGenesis = WinDLL(dll_path)

p = psutil.Process(os.getpid())
dlls = [m.path for m in p.memory_maps() if m.path.lower().endswith('.dll')]

DLL function list:
Microsoft (R) COFF/PE Dumper Version 14.44.35222.0
Copyright (C) Microsoft Corporation.  All rights reserved.


Dump of file D:\source\repos\FnGenesis\x64\Debug\FnGenesis.dll

File Type: DLL

  Section contains the following exports for FnGenesis.dll

    00000000 characteristics
    FFFFFFFF time date stamp
        0.00 version
           1 ordinal base
          39 number of functions
          39 number of names

    ordinal hint RVA      name

          1    0 00395492 ?GetVersionFnG@@YANPEA_WV?$unique_ptr@PEA_WU?$default_delete@PEA_W@std@@@std@@@Z = @ILT+21645(?GetVersionFnG@@YANPEA_WV?$unique_ptr@PEA_WU?$default_delete@PEA_W@std@@@std@@@Z)
          2    1 00391392 ?bdtkillerFC_inner@@YA?AV?$basic_string@DU?$char_traits@D@std@@V?$allocator@D@2@@std@@PEADPEAN@Z = @ILT+5005(?bdtkillerFC_inner@@YA?AV?$basic_string@DU?$char_traits@D@std@@V?$allocator@D@2@@std@@PEADPEAN@Z)
          3    2 00399948 ?crskillerFC_inner@@YA?AV?$basic_string@DU?$char_traits@D@std@@V?$al

**함수명, input/output 자료형 설정**  
: 시스템 호출용 규격인 **"BSTR abcKillerFC(BSTR inputs, BSTR ptr outputs)"** 를 기준으로 설정합니다.

In [38]:
# 함수명 설정
#function = 'MBSjsonFn'
#function = 'DCFFC_sys'
#function = 'zerocallable_calc'
function = 'zckillerFC'
FnGenesis = dllFnGenesis[function]
FnGenesis.argtypes = [c_wchar_p]
#FnGenesis.argtypes = [c_wchar_p, POINTER(c_wchar_p)]
FnGenesis.restype = c_wchar_p

In [39]:
# input json 경로 설정
#input_json_path = "C:/Users/Fnpricing/Documents/fnpricing/[D1050]KR6004943F91문제20260123/[D1050]PM^KR6004943F91^20260123_(문제)input.json"
input_json_path = "ZCKiller/input_54.json"

with open(input_json_path, 'r', encoding='UTF8') as f:
        json_data = json.load(f)
        
# input/output 자료형 설정
input_json_str = str(json_data)
input_json = json.dumps(json_data)
ouput_json = c_wchar_p()
ouput_json_ptr = byref(ouput_json)

**엔진 동작**  
: 프로세스 연결 후 디버깅 시 본 단계에서 소스코드에 진입하여 디버깅 가능합니다.

In [40]:
# 엔진 동작
ouput_json = FnGenesis(input_json)
#ret = FnGenesis(input_json, ouput_json_ptr)
#print(ouput_json_ptr.value)
print(ouput_json)

{"ErrorCode":99001,"ErrorCode_NoInput":"","ErrorCode_WrongInput":"","ErrorCode_Function":"bootstrap_by_fillingYTM Interpolation\n"}


# dll 메모리 해제

테스터를 종료하지 않은 채 소스코드를 수정하고 dll을 다시 빌드하는 경우, 메모리를 해제해주지 않으면 빌드에 실패합니다. (아래 셀들을 실행 후 dll을 다시 빌드하여 "3. dll 실행"으로 이동하면 됩니다.)

In [22]:
# 로딩된 dll의 주소(handle) 확인
print(f"DLL handle: {hex(dllFnGenesis._handle)}")

# FreeLibrary 함수(dll 메모리 해제) input/ouput 자료형 설정
windll.kernel32.FreeLibrary.argtypes = [wintypes.HMODULE]
windll.kernel32.FreeLibrary.restype = wintypes.BOOL

# DLL 핸들 추출
handle = dllFnGenesis._handle

# DLL 해제 (result = 1 이면 성공)
result = windll.kernel32.FreeLibrary(handle)
print(result)

# 'dllFnGenesis' 변수 해제
del dllFnGenesis

NameError: name 'dllFnGenesis' is not defined